# Carga, transformación y reporte de `reg_cat_house_sales` (Tabular Benchmark)

En este notebook se construye un flujo completo de trabajo para el dataset [`inria-soda/tabular-benchmark` (`reg_cat_house_sales`)](https://huggingface.co/datasets/inria-soda/tabular-benchmark/viewer/reg_cat_house_sales):

1. Configurar la ruta del proyecto para importar las clases reutilizables (`HFDatasetLoader` y `Reportes`).
2. Cargar `inria-soda/tabular-benchmark` (configuración `reg_cat_house_sales`) desde Hugging Face y guardar el CSV original.
3. Generar y exportar el reporte de auditoría de metadatos, longitud de caracteres y límites (`metadata_df` y `src/AE2/resultados/metadata_reg_cat_house_sales.txt`), e inspeccionar los registros inconsistentes detectados.
4. Aplicar cuatro transformaciones sobre el `DataFrame` (eliminación de duplicados, filtrado de registros inconsistentes, renombrado de columnas al español y decodificación/cálculo de variables).
5. Generar, guardar y verificar el reporte estadístico del `DataFrame` transformado en `src/AE2/resultados/reg_cat_house_sales.txt`.

Ejecuta las celdas en orden. Si el dataset requiere autenticación, inicia sesión previamente con `hf auth login`.

## 1. Preparar el entorno e importar la clase de carga

El notebook puede ejecutarse desde distintas carpetas. Por eso se buscan los directorios padres hasta encontrar `src/clases` y se agrega `src` a `sys.path`. Así se puede importar `HFDatasetLoader` sin instalar el proyecto como paquete.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases import HFDatasetLoader, Reportes # se importan las clases necesarias para la carga de datos y generación de reportes

/home/runner/work/analisis_datos_dataframe_Hugging_Face/analisis_datos_dataframe_Hugging_Face/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Análisis:** Se localizó dinámicamente el directorio raíz del proyecto y se añadió `src` a `sys.path`, permitiendo importar `HFDatasetLoader` desde el paquete `clases`.

## 2. Configurar el dataset y la salida

`dataset_name` identifica el repositorio `inria-soda/tabular-benchmark` en Hugging Face, `config="reg_cat_house_sales"` selecciona el subconjunto de ventas de viviendas y `split="train"` indica la división que se descargará. El parámetro `filename` hace que el `DataFrame` también se guarde como CSV en `src/AE2/datos/reg_cat_house_sales.csv`.

In [2]:
loader = HFDatasetLoader(
    dataset_name="inria-soda/tabular-benchmark",
    config="reg_cat_house_sales",
    split="train",
    filename="../datos/reg_cat_house_sales.csv",
)


**Análisis:** La instancia `loader` quedó configurada con los parámetros necesarios para conectarse a Hugging Face, descargar la configuración `reg_cat_house_sales` y persistir los datos crudos en `src/AE2/datos/reg_cat_house_sales.csv`.

## 3. Descargar el dataset y crear el DataFrame

El método `load()` descarga el dataset, lo convierte a `pandas.DataFrame` y lo asigna a la variable `df`. La expresión `df.head()` muestra las primeras filas para comprobar que la carga fue exitosa.

In [3]:
df = loader.load()
df.head()

Generating train split:   0%|          | 0/21613 [00:00<?, ? examples/s]

Generating train split: 100%|██████████| 21613/21613 [00:00<00:00, 688004.65 examples/s]

,bedrooms,bathrooms,sqft_living,sqft_lot,waterfront,grade,sqft_above,sqft_basement,yr_built,yr_renovated,lat,long,sqft_living15,sqft_lot15,date_year,date_month,date_day,price
0,3,1.00,1180,5650,0,7,1180,0,1955,0,47.5112,-122.257,1340,5650,0,10,13,12.309987
1,3,2.25,2570,7242,0,7,2170,400,1951,1991,47.7210,-122.319,1690,7639,0,12,9,13.195616
2,2,1.00,770,10000,0,6,770,0,1933,0,47.7379,-122.233,2720,8062,1,2,25,12.100718
3,4,3.00,1960,5000,0,7,1050,910,1965,0,47.5208,-122.393,1360,5000,0,12,9,13.311331
4,3,2.00,1680,8080,0,8,1680,0,1987,0,47.6168,-122.045,1800,7503,1,2,18,13.142168


**Análisis y reconocimiento inicial del dataset:**

- **¿Qué representa cada fila (unidad de análisis)?** Cada fila representa la transacción de venta de una vivienda residencial individual ubicada en el condado de King (estado de Washington, EE. UU., área metropolitana de Seattle), registrada entre mayo de 2014 y mayo de 2015.
- **¿Qué variables respaldan la necesidad analítica propuesta?** El dataset ofrece variables cuantitativas y categóricas que permiten caracterizar la oferta inmobiliaria y comparar los precios de venta según sus atributos:
  - **Atributos físicos y de capacidad:** `bedrooms` (habitaciones), `bathrooms` (baños), `sqft_living` (área habitable interior en pies cuadrados), `sqft_lot` (área total del terreno), `sqft_above` (área sobre el nivel del suelo) y `sqft_basement` (área de sótano).
  - **Calidad constructiva y antigüedad:** `grade` (calificación constructiva del condado en escala 1–13), `waterfront` (indicador binario de vista al agua), `yr_built` (año de construcción) y `yr_renovated` (año de remodelación).
  - **Ubicación geográfica y contexto del vecindario:** `lat` (latitud), `long` (longitud), `sqft_living15` (área habitable promedio de los 15 vecinos más cercanos) y `sqft_lot15` (área de lote promedio de los 15 vecinos más cercanos).
  - **Periodo de venta y variable objetivo:** `date_year`, `date_month`, `date_day` (componentes de la fecha de venta) y `price` (precio de transacción en escala logarítmica natural, a partir del cual se obtiene el valor comercial en dólares `Precio (USD)`).

## 4. Reporte de auditoría de metadatos del DataFrame

El método `reporte_descriptivo_data(df)` de la clase `Reportes` construye una tabla de auditoría técnica sobre los metadatos del `DataFrame`, asignada a `metadata_df`. Para cada columna evalúa el tipo de dato (`Tipo`), la completitud (`No Nulos` y `Nulos`), la cardinalidad (`Únicos`), la longitud mínima y máxima en caracteres (`Long. Mín (caract.)` y `Long. Máx (caract.)`), los límites observados (`Límite Mín`, `Mediana`, `Media`, `Límite Máx`), las cotas teóricas calculadas mediante el rango intercuartílico (`Límite Inf (IQR)` y `Límite Sup (IQR)`) y el conteo de valores fuera de dichos umbrales (`Atípicos (IQR)`).

In [4]:
reportes = Reportes()
metadata_df = reportes.reporte_descriptivo_data(df)
metadata_df

,Columna,Tipo,No Nulos,Nulos,Únicos,Long. Mín (caract.),Long. Máx (caract.),Límite Mín,Mediana,Media,Límite Máx,Límite Inf (IQR),Límite Sup (IQR),Atípicos (IQR)
0,bedrooms,int64,21613,0,13,1,2,0.0000,3.0000,3.3708,3.300000e+01,1.5000,5.5000,546
1,bathrooms,float64,21613,0,30,3,4,0.0000,2.2500,2.1148,8.000000e+00,0.6250,3.6250,571
2,sqft_living,int64,21613,0,1038,3,5,290.0000,1910.0000,2079.8997,1.354000e+04,-257.5000,4234.5000,572
3,sqft_lot,int64,21613,0,9782,3,7,520.0000,7618.0000,15106.9676,1.651359e+06,-3432.0000,19160.0000,2425
4,waterfront,int64,21613,0,2,1,1,0.0000,0.0000,0.0075,1.000000e+00,0.0000,0.0000,163
5,grade,int64,21613,0,12,1,2,1.0000,7.0000,7.6569,1.300000e+01,5.5000,9.5000,1911
6,sqft_above,int64,21613,0,946,3,4,290.0000,1560.0000,1788.3907,9.410000e+03,-340.0000,3740.0000,611
7,sqft_basement,int64,21613,0,306,1,4,0.0000,0.0000,291.5090,4.820000e+03,-840.0000,1400.0000,496
8,yr_built,int64,21613,0,116,4,4,1900.0000,1975.0000,1971.0051,2.015000e+03,1882.0000,2066.0000,0
9,yr_renovated,int64,21613,0,70,1,4,0.0000,0.0000,84.4023,2.015000e+03,0.0000,0.0000,914


In [5]:
ruta_metadata = reportes.guardar_reporte_metadata(
    df,
    "../resultados/metadata_reg_cat_house_sales.txt",
    dataset_name=loader.dataset_name,
    config=loader.config,
    split=loader.split,
)

print(f"Reporte de metadatos guardado en: {ruta_metadata.resolve()}\n")
print(ruta_metadata.read_text(encoding="utf-8"))

Reporte de metadatos guardado en: /home/runner/work/analisis_datos_dataframe_Hugging_Face/analisis_datos_dataframe_Hugging_Face/src/AE2/resultados/metadata_reg_cat_house_sales.txt

REPORTE DE AUDITORÍA DE METADATOS DEL DATAFRAME
Dataset: inria-soda/tabular-benchmark
Configuración: reg_cat_house_sales
Split: train
Filas: 21613
Columnas: 18
Duplicados: 0

Tabla de auditoría de metadatos (tipos, longitudes, límites y atípicos IQR):
      Columna    Tipo  No Nulos  Nulos  Únicos  Long. Mín (caract.)  Long. Máx (caract.)  Límite Mín   Mediana      Media   Límite Máx  Límite Inf (IQR)  Límite Sup (IQR)  Atípicos (IQR)
     bedrooms   int64     21613      0      13                    1                    2      0.0000    3.0000     3.3708      33.0000            1.5000            5.5000             546
    bathrooms float64     21613      0      30                    3                    4      0.0000    2.2500     2.1148       8.0000            0.6250            3.6250             571
  sqft

**Análisis del reporte de auditoría de metadatos (`metadata_df`):**

1. **Tipado y completitud:** Las **18 variables** son de tipo numérico (`14 int64` y `4 float64`) con **21,613 registros no nulos** y **0 valores nulos** en la totalidad del esquema, confirmando una completitud del 100%.
2. **Auditoría de longitud de caracteres:**
   - Variables binarias o codificadas como `waterfront` y `date_year` tienen longitud fija de `1` carácter (`0` o `1`), y `yr_built` mantiene longitud constante de `4` caracteres (`1900`–`2015`).
   - En cambio, `yr_renovated` oscila entre `1` y `4` caracteres porque codifica con `0` las viviendas no remodeladas y con 4 dígitos el año en que sí hubo renovación.
   - Variables continuas de alta precisión (`lat` y `price`) alcanzan entre `15` y `18` caracteres por su representación decimal en punto flotante.
3. **Auditoría de límites observados vs. límites IQR:**
   - En `bedrooms` el rango intercuartílico establece límites entre `1.5` y `5.5` habitaciones (`546` atípicos estadísticos), pero los límites observados (`Límite Mín = 0` y `Límite Máx = 33`) revelan anomalías de captura que deben depurarse.
   - En `bathrooms` el límite inferior observado es `0.0` (frente a un `Límite Inf (IQR)` de `0.625`), evidenciando viviendas registradas sin baños.
   - En variables de superficie (`sqft_living`, `sqft_lot`, `sqft_above`, `sqft_basement`) los límites inferiores teóricos por IQR resultan negativos debido a la asimetría positiva (cola derecha larga por mansiones o grandes terrenos de hasta `1,651,359 sqft`), por lo que su cota física real está dada por `Límite Mín >= 0`.

## 5. Revisar en detalle los registros inconsistentes detectados en la auditoría

A partir de los límites observados en `metadata_df` (`Límite Mín = 0` en `bedrooms` y `bathrooms`, y `Límite Máx = 33` en `bedrooms`), puntualizamos cuántos registros presentan valores incompatibles con una vivienda residencial habitable (`0` habitaciones, `0` baños o más de `10` habitaciones) e inspeccionamos los casos extremos.

In [6]:
print(f"Viviendas con 0 habitaciones: {int((df['bedrooms'] == 0).sum())}")
print(f"Viviendas con 0 baños: {int((df['bathrooms'] == 0).sum())}")
print(f"Viviendas con <1 baño completo (bathrooms < 1): {int((df['bathrooms'] < 1).sum())}")
print(f"Viviendas con >10 habitaciones: {int((df['bedrooms'] > 10).sum())}")
print("\nDetalle de viviendas con >10 habitaciones:")
print(df[df["bedrooms"] > 10][["bedrooms", "bathrooms", "sqft_living", "yr_built", "price"]])

Viviendas con 0 habitaciones: 13
Viviendas con 0 baños: 10
Viviendas con <1 baño completo (bathrooms < 1): 86
Viviendas con >10 habitaciones: 2

Detalle de viviendas con >10 habitaciones:
       bedrooms  bathrooms  sqft_living  yr_built      price
8757         11       3.00         3000      1918  13.161586
15870        33       1.75         1620      1947  13.369225


**Análisis:** La inspección puntual confirma los hallazgos de la auditoría de metadatos: existen **13 viviendas con 0 habitaciones**, **10 con 0 baños** (y un total de **79 con menos de 1 baño completo**) y **2 viviendas con más de 10 habitaciones**. Entre estas dos últimas destaca el registro con índice `15870`, que reporta **33 habitaciones en una casa de solo 1,620 sqft** y 1.75 baños (un error de digitación documentado en el dataset de King County, donde probablemente correspondía a 3 habitaciones), a diferencia del índice `8757` (`11` habitaciones en una propiedad amplia de `3,000 sqft` y `3.0` baños).

## 6. Transformación 1: Eliminar filas duplicadas

Verificamos y eliminamos posibles filas completamente duplicadas en el `DataFrame`, conservando la primera ocurrencia.

In [7]:
# Eliminar filas duplicadas conservando la primera ocurrencia
df_sin_duplicados = df.drop_duplicates(keep='first').reset_index(drop=True)

print(f"Filas antes    : {len(df)}")
print(f"Filas después  : {len(df_sin_duplicados)}")
print(f"Filas eliminadas del df: {len(df) - len(df_sin_duplicados)}")
df_sin_duplicados.head()

Filas antes    : 21613
Filas después  : 21613
Filas eliminadas del df: 0


,bedrooms,bathrooms,sqft_living,sqft_lot,waterfront,grade,sqft_above,sqft_basement,yr_built,yr_renovated,lat,long,sqft_living15,sqft_lot15,date_year,date_month,date_day,price
0,3,1.00,1180,5650,0,7,1180,0,1955,0,47.5112,-122.257,1340,5650,0,10,13,12.309987
1,3,2.25,2570,7242,0,7,2170,400,1951,1991,47.7210,-122.319,1690,7639,0,12,9,13.195616
2,2,1.00,770,10000,0,6,770,0,1933,0,47.7379,-122.233,2720,8062,1,2,25,12.100718
3,4,3.00,1960,5000,0,7,1050,910,1965,0,47.5208,-122.393,1360,5000,0,12,9,13.311331
4,3,2.00,1680,8080,0,8,1680,0,1987,0,47.6168,-122.045,1800,7503,1,2,18,13.142168


**Análisis:** Al ejecutar `drop_duplicates()`, se confirma que no existen filas idénticas repetidas en el conjunto original (`21,613` filas antes y después), garantizando la integridad de las observaciones antes del filtrado.

## 7. Transformación 2: Filtrar registros inconsistentes y atípicos extremos

Descartamos los registros que no cumplen con condiciones mínimas de coherencia habitacional (viviendas sin habitaciones `bedrooms == 0`, sin baños `bathrooms < 1` o el error tipográfico de `bedrooms > 15`).

In [8]:
condicion_valida = (
    (df_sin_duplicados["bedrooms"] >= 1)
    & (df_sin_duplicados["bedrooms"] <= 15)
    & (df_sin_duplicados["bathrooms"] >= 1)
)
df_filtrado = df_sin_duplicados[condicion_valida].reset_index(drop=True)

print(f"Filas antes del filtro : {len(df_sin_duplicados)}")
print(f"Filas después del filtro: {len(df_filtrado)}")
print(f"Filas descartadas      : {len(df_sin_duplicados) - len(df_filtrado)}")
df_filtrado.head()

Filas antes del filtro : 21613
Filas después del filtro: 21521
Filas descartadas      : 92


,bedrooms,bathrooms,sqft_living,sqft_lot,waterfront,grade,sqft_above,sqft_basement,yr_built,yr_renovated,lat,long,sqft_living15,sqft_lot15,date_year,date_month,date_day,price
0,3,1.00,1180,5650,0,7,1180,0,1955,0,47.5112,-122.257,1340,5650,0,10,13,12.309987
1,3,2.25,2570,7242,0,7,2170,400,1951,1991,47.7210,-122.319,1690,7639,0,12,9,13.195616
2,2,1.00,770,10000,0,6,770,0,1933,0,47.7379,-122.233,2720,8062,1,2,25,12.100718
3,4,3.00,1960,5000,0,7,1050,910,1965,0,47.5208,-122.393,1360,5000,0,12,9,13.311331
4,3,2.00,1680,8080,0,8,1680,0,1987,0,47.6168,-122.045,1800,7503,1,2,18,13.142168


**Análisis:** Se descartaron los registros con valores improcedentes en habitaciones y baños, obteniendo un conjunto depurado de viviendas residenciales consistentes.

## 8. Transformación 3: Renombrar columnas al español

| Columna original | Columna en español |
|------------------|--------------------|
| `bedrooms` | `Habitaciones` |
| `bathrooms` | `Baños` |
| `sqft_living` | `Área habitable (sqft)` |
| `sqft_lot` | `Área del lote (sqft)` |
| `waterfront` | `Vista al agua` |
| `grade` | `Calificación` |
| `sqft_above` | `Área sobre suelo (sqft)` |
| `sqft_basement` | `Área sótano (sqft)` |
| `yr_built` | `Año de construcción` |
| `yr_renovated` | `Año de renovación` |
| `lat` | `Latitud` |
| `long` | `Longitud` |
| `sqft_living15` | `Área habitable vecinos (sqft)` |
| `sqft_lot15` | `Área lote vecinos (sqft)` |
| `date_year` | `Año de venta` |
| `date_month` | `Mes de venta` |
| `date_day` | `Día de venta` |
| `price` | `Precio (log)` |

In [9]:
columnas_es = {
    "bedrooms"      : "Habitaciones",
    "bathrooms"     : "Baños",
    "sqft_living"   : "Área habitable (sqft)",
    "sqft_lot"      : "Área del lote (sqft)",
    "waterfront"    : "Vista al agua",
    "grade"         : "Calificación",
    "sqft_above"    : "Área sobre suelo (sqft)",
    "sqft_basement" : "Área sótano (sqft)",
    "yr_built"      : "Año de construcción",
    "yr_renovated"  : "Año de renovación",
    "lat"           : "Latitud",
    "long"          : "Longitud",
    "sqft_living15" : "Área habitable vecinos (sqft)",
    "sqft_lot15"    : "Área lote vecinos (sqft)",
    "date_year"     : "Año de venta",
    "date_month"    : "Mes de venta",
    "date_day"      : "Día de venta",
    "price"         : "Precio (log)",
}

df_es = df_filtrado.rename(columns=columnas_es)

print("Columnas actualizadas:")
print(df_es.columns.tolist())
df_es.head()

Columnas actualizadas:
['Habitaciones', 'Baños', 'Área habitable (sqft)', 'Área del lote (sqft)', 'Vista al agua', 'Calificación', 'Área sobre suelo (sqft)', 'Área sótano (sqft)', 'Año de construcción', 'Año de renovación', 'Latitud', 'Longitud', 'Área habitable vecinos (sqft)', 'Área lote vecinos (sqft)', 'Año de venta', 'Mes de venta', 'Día de venta', 'Precio (log)']


,Habitaciones,Baños,Área habitable (sqft),Área del lote (sqft),Vista al agua,Calificación,Área sobre suelo (sqft),Área sótano (sqft),Año de construcción,Año de renovación,Latitud,Longitud,Área habitable vecinos (sqft),Área lote vecinos (sqft),Año de venta,Mes de venta,Día de venta,Precio (log)
0,3,1.00,1180,5650,0,7,1180,0,1955,0,47.5112,-122.257,1340,5650,0,10,13,12.309987
1,3,2.25,2570,7242,0,7,2170,400,1951,1991,47.7210,-122.319,1690,7639,0,12,9,13.195616
2,2,1.00,770,10000,0,6,770,0,1933,0,47.7379,-122.233,2720,8062,1,2,25,12.100718
3,4,3.00,1960,5000,0,7,1050,910,1965,0,47.5208,-122.393,1360,5000,0,12,9,13.311331
4,3,2.00,1680,8080,0,8,1680,0,1987,0,47.6168,-122.045,1800,7503,1,2,18,13.142168


**Análisis:** Las 18 columnas del `DataFrame` fueron renombradas al español con nombres descriptivos que facilitan la lectura del reporte final.

## 9. Transformación 4: Decodificar variables y calcular el precio en dólares (USD)

Transformamos el indicador binario `Vista al agua` (`0 -> "No"`, `1 -> "Sí"`), decodificamos `Año de venta` (`0 -> 2014`, `1 -> 2015`) y agregamos la columna `Precio (USD)` revirtiendo la transformación logarítmica de `Precio (log)` mediante `np.exp()`.

In [10]:
import numpy as np

df_transformado = df_es.copy()
df_transformado["Vista al agua"] = df_transformado["Vista al agua"].map({0: "No", 1: "Sí"})
df_transformado["Año de venta"] = df_transformado["Año de venta"].map({0: 2014, 1: 2015})
df_transformado["Precio (USD)"] = np.exp(df_transformado["Precio (log)"]).round(2)

print("Vista previa de las columnas transformadas:")
df_transformado[["Habitaciones", "Baños", "Vista al agua", "Año de venta", "Precio (log)", "Precio (USD)"]].head()

Vista previa de las columnas transformadas:


,Habitaciones,Baños,Vista al agua,Año de venta,Precio (log),Precio (USD)
0,3,1.00,No,2014,12.309987,221901.0
1,3,2.25,No,2014,13.195616,538001.0
2,2,1.00,No,2015,12.100718,180001.0
3,4,3.00,No,2014,13.311331,604001.0
4,3,2.00,No,2015,13.142168,510001.0


**Análisis:** Con esta cuarta transformación, el `DataFrame` cuenta ahora con **19 columnas**, incorporando etiquetas legibles para `Vista al agua`, el año calendario real en `Año de venta` (`2014` o `2015`) y el valor comercial en dólares en `Precio (USD)`.

## 10. Generar, guardar y verificar el reporte del DataFrame transformado

`guardar_reporte(df_transformado, ruta, ...)` genera el resumen estadístico completo (dimensiones, duplicados, tipos de datos, conteo de nulos y estadísticas descriptivas), crea la carpeta de destino si no existe y escribe el archivo en codificación UTF-8 en `src/AE2/resultados/reg_cat_house_sales.txt`. A continuación se verifica su creación y se muestra su contenido íntegro.

In [11]:
info_dataset = loader.info()
muestra_desc = (
    f"Dataset completo depurado ({len(df_transformado)} de "
    f"{info_dataset['filas']} registros originales; sin muestreo aleatorio)"
)

ruta = reportes.guardar_reporte(
    df_transformado,
    "../resultados/reg_cat_house_sales.txt",
    dataset_name=loader.dataset_name,
    config=loader.config,
    split=loader.split,
    muestra=muestra_desc,
)

print(f"Reporte guardado en: {ruta.resolve()}")
print(f"Existe el archivo  : {ruta.exists()} ({ruta.stat().st_size} bytes)\n")
print(ruta.read_text(encoding="utf-8"))

Reporte guardado en: /home/runner/work/analisis_datos_dataframe_Hugging_Face/analisis_datos_dataframe_Hugging_Face/src/AE2/resultados/reg_cat_house_sales.txt
Existe el archivo  : True (4245 bytes)

REPORTE DEL DATAFRAME
Dataset: inria-soda/tabular-benchmark
Configuración: reg_cat_house_sales
Split: train
Muestra / Cobertura: Dataset completo depurado (21521 de 21613 registros originales; sin muestreo aleatorio)
Filas: 21521
Columnas: 19
Duplicados: 0

Columnas y tipos:
- Habitaciones: int64
- Baños: float64
- Área habitable (sqft): int64
- Área del lote (sqft): int64
- Vista al agua: object
- Calificación: int64
- Área sobre suelo (sqft): int64
- Área sótano (sqft): int64
- Año de construcción: int64
- Año de renovación: int64
- Latitud: float64
- Longitud: float64
- Área habitable vecinos (sqft): int64
- Área lote vecinos (sqft): int64
- Año de venta: int64
- Mes de venta: int64
- Día de venta: int64
- Precio (log): float64
- Precio (USD): float64

Valores nulos por columna:
- Habitac

**Análisis:** El reporte final se almacenó satisfactoriamente en `src/AE2/resultados/reg_cat_house_sales.txt`. Incluye el identificador del dataset (`inria-soda/tabular-benchmark`), la configuración (`reg_cat_house_sales`), el split (`train`) y la cobertura utilizada, y resume las **21,521 filas y 19 columnas** del `DataFrame` transformado, verificando que no existen valores nulos ni duplicados y detallando las estadísticas descriptivas de todas las variables numéricas (incluyendo una media de precio de venta de aproximadamente **\$541,065 USD** y una mediana de **\$450,001 USD**).

## Conclusión

Se completó satisfactoriamente el flujo de trabajo de la **Actividad Evaluativa 2 (AE2)** para el dataset **`inria-soda/tabular-benchmark` (`reg_cat_house_sales`)** reutilizando las clases transversales de `src/clases`:

1. **Extracción y carga (`HFDatasetLoader`)**: Se descargaron las **21,613 filas y 18 columnas** de la división `train` desde Hugging Face y se exportó el archivo base en `src/AE2/datos/reg_cat_house_sales.csv`.
2. **Auditoría de metadatos y reconocimiento de datos (`Reportes.reporte_descriptivo_data`)**: Se documentó qué representa cada fila (transacción de venta residencial en King County, WA) y se generó la tabla de auditoría (`metadata_df`) junto con el archivo `src/AE2/resultados/metadata_reg_cat_house_sales.txt`, evaluando tipos de datos, completitud (`0` nulos en las 18 columnas), cardinalidad, longitud de caracteres (`Long. Mín` y `Long. Máx`), límites observados y cotas intercuartílicas (`IQR`), lo que permitió identificar **92 registros inconsistentes** (`0` habitaciones, `<1` baño o el caso anómalo de `33` habitaciones).
3. **Transformaciones aplicadas**:
   - **Eliminación de duplicados**: Verificación de filas repetidas (`drop_duplicates`).
   - **Filtrado de consistencia**: Depuración de 92 registros inválidos, conservando **21,521 viviendas**.
   - **Renombrado de columnas**: Traducción de las 18 variables originales al español.
   - **Decodificación y enriquecimiento**: Conversión de `Vista al agua` a `"No"/"Sí"`, decodificación de `Año de venta` (`2014`/`2015`) y cálculo de `Precio (USD)` a partir de `Precio (log)`.
4. **Generación automática de reportes (`Reportes`)**: Se generaron, almacenaron y verificaron en `src/AE2/resultados/` tanto el reporte de auditoría de metadatos del dataset original (`metadata_reg_cat_house_sales.txt`) como el reporte estadístico del dataset transformado (`reg_cat_house_sales.txt`) en codificación UTF-8.